# Thesis

In [ ]:
import requests
import pandas as pd
import praw
import pandas as pd
import praw

## Code using the YT video

In [ ]:
CLIENT_ID = "Your cliend id here"
SECRET_KEY = "The secret key here"

In [ ]:
auth = requests.auth.HTTPBasicAuth(CLIENT_ID, SECRET_KEY)

In [ ]:
data = {
    'grant_type': 'password',
    'username': 'your_username',
    'password': 'your_password'
    }

In [ ]:
headers = {'User-Agent': 'MyApi/0.0.1'}

In [ ]:
res = requests.post('https://www.reddit.com/api/v1/access_token',
                    auth=auth,
                    data=data,
                    headers=headers)

In [ ]:
TOKEN = res.json()["access_token"]

In [ ]:
headers = {**headers, **{'Authorization': f"bearer {TOKEN}"}}

In [ ]:
headers

In [ ]:
requests.get("https://oauth.reddit.com/api/v1/me",
             headers=headers,
             )

In [ ]:
requests.get("https://oauth.reddit.com/api/v1/me",
             headers=headers,
             ).json()

In [ ]:
res = requests.get('https://oauth.reddit.com/r/python/hot',
                 headers=headers,
                 params={'limit': '100'}
                 ) 

# Python is the Subreddit name.
# If I would like to change it then I would change the name after /r/
# Attention, in this way we have access to the HOT posts, but maybe we are 
# interested in the NEW posts, so we would change 'hot' with 'new'

In [ ]:
for post in res.json()['data']['children']:
    print(post['data'])

In [ ]:
post['data'].keys()

In [ ]:
rows = []

for post in res.json()['data']['children']:
    rows.append({
        'SubReddit': post['data']['subreddit'],
        'Title': post['data']['title'],
        'Text': post['data']['selftext'],
        'Score': post['data']['score'],
        'UpVote_Ratio': post['data']['upvote_ratio'],
        'ups': post['data']['ups'],
        'downs': post['data']['downs'],
        'Num_Comments': post['data']['num_comments'],
        "created_utc": post['data']['created_utc'],
    })

df = pd.DataFrame(rows)

In [ ]:
df

In [ ]:
res = requests.get('https://oauth.reddit.com/r/python/new',
                 headers=headers
                 ) 

In [ ]:
rows = []

for post in res.json()['data']['children']:
    rows.append({
        'SubReddit': post['data']['subreddit'],
        'Title': post['data']['title'],
        'Text': post['data']['selftext'],
        'Score': post['data']['score'],
        'UpVote_Ratio': post['data']['upvote_ratio'],
        'ups': post['data']['ups'],
        'downs': post['data']['downs'],
        'Num_Comments': post['data']['num_comments']
    })

df = pd.DataFrame(rows)

In [ ]:
df

Il problema più grande è che con questo codice si riesce a prendere tranquillamente i post, ma non trovo dei video che permettono di ottenere anche i commenti

In [ ]:
reddit = praw.Reddit(
    client_id="your_client_id",
    client_secret="your_client_secret",
    user_agent="MyApi/0.0.1",
    username="your_username",
    password="your_password"
)

In [ ]:
subreddit = reddit.subreddit("Futurology")
post_rows = []
comment_rows = []

for post in subreddit.hot(limit=100):
    post_rows.append({
        'post_id': post.id,
        'SubReddit': post.subreddit.display_name,
        'Title': post.title,
        'Score': post.score,
        'UpVote_Ratio': post.upvote_ratio,
        'Num_Comments': post.num_comments,
        "created_utc": post.created_utc,
    })

    post.comments.replace_more(limit=0) 
    for comment in post.comments.list():
        if isinstance(comment, praw.models.Comment):
            comment_rows.append({
                'post_id': post.id, 
                'comment_id': comment.id,
                'author': str(comment.author), 
                'body': comment.body,
                'score': comment.score,
            })

## Using PRAW such as Gemini 2.5 suggested - NEW

In [ ]:
reddit = praw.Reddit(
    client_id="your_client_id",
    client_secret="your_client_secret",
    user_agent="MyApi/0.0.1",
    username="your_username",
    password="your_password"
)

In [ ]:
subreddit = reddit.subreddit("Futurology")
post_rows = []
comment_rows = []

for post in subreddit.new(limit=100):
    post_rows.append({
        'post_id': post.id,
        'SubReddit': post.subreddit.display_name,
        'Title': post.title,
        'Score': post.score,
        'UpVote_Ratio': post.upvote_ratio,
        'Num_Comments': post.num_comments,
        "created_utc": post.created_utc,
    })

    post.comments.replace_more(limit=0) 
    for comment in post.comments.list():
        if isinstance(comment, praw.models.Comment):
            comment_rows.append({
                'post_id': post.id, 
                'comment_id': comment.id,
                'author': str(comment.author), 
                'body': comment.body,
                'score': comment.score,
            })

## Using PRAW such as Gemini 2.5 suggested - search!

PRAW sta per Python Reddit API Weapper ed è un libreria che semplifica l'accesso all'API di reddit. Questa libreria consente quindi di accedere in maniera rapida di scaricare informazioni dai post o commenti dei reddit. Per questo motivo utile per raccogliere grandi dati da Reddit ed utilizzarli per procedere con degli studi tra cui la sentiment analysis

Prima di tutto procedo inserendo le mie credenziali API e di Reddit (Attenzione, nella tesi / GitHub nascondile se no possono rubarti l'acc)

In [ ]:
reddit = praw.Reddit(
    client_id="your_client_id",
    client_secret="your_client_secret",
    user_agent="MyApi/0.0.1",
    username="your_username",
    password="your_password"
)

/Users/carlo_air_2021/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Il subreddit r/Futurology è stato scelto in quanto dedicato principalmente a discussioni sul futuro, con particolare attenzione alle tecnologie emergenti, alla scienza e alla ricerca, ma anche ad aspetti etici, filosofici e sociali. Si tratta quindi di una comunità molto attiva, rendendo questo subreddit ricco di dati, inoltre spesso le discussioni al suo interno si concentrano su scenari futuri legati all’intelligenza artificiale.

Per rendere l’analisi più mirata agli obiettivi della ricerca, la raccolta dei dati è stata filtrata in base a quattro temi principali, ciascuno dei quali associato a un insieme di parole chiave:
- AGI e Sviluppo Tecnologico: AGI, general intelligence, GPT, deep learning, neural network, AI
- Regolamentazione: AI Act, regulation, policy, law
- Impatto sul lavoro: jobs, employment, workforce, automation
- Sicurezza: safety, risk, existential, danger
- Etica e AI: ethics, ethical, bias, fairness, moral

Tramite un ciclo for sono stati raccolti 50 post per ciascun tema (per un totale di 200 post), insieme a tutti i commenti collegati a tali post, così da poter includere non solo il contenuto principale ma anche le opinioni degli utenti in merito.

In [ ]:
subreddit = reddit.subreddit("Futurology")
all_posts = []
all_comments = []

ai_core_query = 'AGI OR "general intelligence" OR GPT OR "deep learning" OR "neural network" OR AI'

temi_di_ricerca = {
    "AGI e Sviluppo Tecnologico": ai_core_query,
    "Regolamentazione e AI": f'("AI Act" OR regulation OR policy OR law) AND ({ai_core_query})',
    "Impatto sul Lavoro e AI": f'(jobs OR employment OR workforce OR automation) AND ({ai_core_query})',
    "Sicurezza e AI": f'(safety OR risk OR existential OR danger) AND ({ai_core_query})',
    "Etica e AI": f'(ethics OR ethical OR bias OR fairness OR moral) AND ({ai_core_query})'
}



In [4]:
for tema, query in temi_di_ricerca.items():
    print(f"\n--- Inizio raccolta per il tema: {tema} ---")
    
    for post in subreddit.search(query, sort='top', time_filter='year', limit=50):
        all_posts.append({
            'search_topic': tema,
            'post_id': post.id,
            'SubReddit': post.subreddit.display_name,
            'Title': post.title,
            'Score': post.score,
            'UpVote_Ratio': post.upvote_ratio,
            'Num_Comments': post.num_comments,
            "created_utc": post.created_utc,
        })
        
        post.comments.replace_more(limit=0)
        for comment in post.comments.list():
            if isinstance(comment, praw.models.Comment):
                all_comments.append({
                    'post_id': post.id,
                    'search_topic': tema, 
                    'comment_id': comment.id,
                    'author': str(comment.author),
                    'body': comment.body,
                    'score': comment.score,
                })


--- Inizio raccolta per il tema: AGI e Sviluppo Tecnologico ---

--- Inizio raccolta per il tema: Regolamentazione e AI ---

--- Inizio raccolta per il tema: Impatto sul Lavoro e AI ---

--- Inizio raccolta per il tema: Sicurezza e AI ---

--- Inizio raccolta per il tema: Etica e AI ---


Creo un dataset per il post e uno per i commenti

In [5]:
posts_df = pd.DataFrame(all_posts)
comments_df = pd.DataFrame(all_comments)

print("DataFrame dei Post:")
print(posts_df.head())
print("\nDataFrame dei Commenti:")
print(comments_df.head())

DataFrame dei Post:
                 search_topic  post_id   SubReddit  \
0  AGI e Sviluppo Tecnologico  1h8h483  Futurology   
1  AGI e Sviluppo Tecnologico  1js17yn  Futurology   
2  AGI e Sviluppo Tecnologico  1lmk3ih  Futurology   
3  AGI e Sviluppo Tecnologico  1lxvkse  Futurology   
4  AGI e Sviluppo Tecnologico  1mrqrt9  Futurology   

                                               Title  Score  UpVote_Ratio  \
0  Murdered Insurance CEO Had Deployed an AI to A...  99503          0.90   
1  White House Accused of Using ChatGPT to Create...  36020          0.94   
2  Bernie Sanders says that if AI makes us so pro...  34816          0.95   
3  Elon: “We tweaked Grok.” Grok: “Call me MechaH...  26039          0.92   
4  AI experts return from China stunned: The U.S....  22905          0.91   

   Num_Comments   created_utc  
0          3523  1.733534e+09  
1           867  1.743852e+09  
2           843  1.751108e+09  
3           957  1.752309e+09  
4          1633  1.755338e+09  


Creo un dataset merged tra post e commenti

In [6]:
merged_df = pd.merge(
    comments_df, 
    posts_df, 
    on='post_id', 
    how='left',
    suffixes=('_comment', '_post')
)

print("\nDataFrame Unito (prime 5 righe):")
print(merged_df.head())


DataFrame Unito (prime 5 righe):
   post_id        search_topic_comment comment_id        author  \
0  1h8h483  AGI e Sviluppo Tecnologico    m0t34l9    mysteryiii   
1  1h8h483  AGI e Sviluppo Tecnologico    m0sx40t   kid_entropy   
2  1h8h483  AGI e Sviluppo Tecnologico    m0t5w7w       Syrairc   
3  1h8h483  AGI e Sviluppo Tecnologico    m0t7ung     austindsb   
4  1h8h483  AGI e Sviluppo Tecnologico    m0syjyw  XtremelyMeta   

                                                body  score  \
0  We get penalized for not having health insuran...   1582   
1  This whole thing feels like it's right out of ...   1456   
2  My favorite thing is the insurance companies c...   6743   
3  I 100% believe this, I have UHC as my provider...    180   
4  I mean, this is what dissolution of the social...  16727   

            search_topic_post   SubReddit  \
0  AGI e Sviluppo Tecnologico  Futurology   
1  AGI e Sviluppo Tecnologico  Futurology   
2  AGI e Sviluppo Tecnologico  Futurology   
3  A

Infine salvo i 3 dataset creati

In [7]:
Percorso_post = "/Users/carlo_air_2021/Desktop/Reddit_Post_ChatGPT.csv"
posts_df.to_csv(Percorso_post, index=False, encoding='utf-8-sig') 
print(f"File salvato con successo in: {Percorso_post}")

File salvato con successo in: /Users/carlo_air_2021/Desktop/Reddit_Post_ChatGPT.csv


In [8]:
Percorso_comments = "/Users/carlo_air_2021/Desktop/Reddit_Comments_ChatGPT.csv"
comments_df.to_csv(Percorso_comments, index=False, encoding='utf-8-sig') 
print(f"File salvato con successo in: {Percorso_comments}")

File salvato con successo in: /Users/carlo_air_2021/Desktop/Reddit_Comments_ChatGPT.csv


In [9]:
Percorso = "/Users/carlo_air_2021/Desktop/Reddit_Merged_ChatGPT.csv"
merged_df.to_csv(Percorso, index=False, encoding='utf-8-sig') 
print(f"File salvato con successo in: {Percorso}")

File salvato con successo in: /Users/carlo_air_2021/Desktop/Reddit_Merged_ChatGPT.csv


### EXTRA

In [10]:
numero_duplicati_post = posts_df['Title'].duplicated().sum()

print(numero_duplicati_post)

44


In [11]:
posts_df = posts_df.drop_duplicates(subset='Title')

In [12]:
numero_duplicati_commenti = comments_df['body'].duplicated().sum()

print(numero_duplicati_commenti)

13667


In [13]:
comments_df = comments_df.drop_duplicates(subset='body')

In [14]:
merged_df.shape

(86731, 13)

In [15]:
merged_df.groupby('search_topic_post').size()

search_topic_post
AGI e Sviluppo Tecnologico    28849
Etica e AI                     5262
Impatto sul Lavoro e AI       25407
Regolamentazione e AI         15419
Sicurezza e AI                11794
dtype: int64